## Nivell 1 
# Exercici 1: 
* Importa com un DataFrame l'arxiu sprint10.xlsx. Assegura't que el fitxer s'importa correctament, amb els noms de columnes que li corresponen, sense manipular l'arxiu original.

In [ ]:
#Exercici 1

import pandas as pd

df = pd.read_excel(r"C:\Users\User\Desktop\sprint10.xlsx", header = 3, index_col=0)
df

* Ordena el DataFrame pel país d'origen. En cas d'empat, ordena pel nom de la ciutat.
* Mostra les primeres 10 files.
* fes un print on comprovi que el DNI només té valors únics.

In [ ]:
#Exercici 1

df.sort_values(by=["País d'origen", "Ciutat"])
df.head(10)
print(df['DNI'].is_unique) 
print(df['DNI'].duplicated().sum())

# Exercici 2: 
* Crea una columna que sigui el nom complet.
* Crea una columna si la persona és nascuda a Espanya o no. 
* Posa el DNI com a índex del DF (noms de files)
* Substitueix el nom de les columnes Dia de Naixement, Mes de Naixement i Any de Naixement per Dia, Mes i Any. 
* Substitueix H per Home, D per Dona, A per Altres i NC per una dada faltant (nan/null/na)
* Mostra tots els canvis en una sola taula.

In [ ]:
#Exercici 2

df_ex2 = df.copy() #creem una còpia del df original pero no modificar-lo directament. 

df_ex2['Nom complet'] = df_ex2['Nom'] + ' ' + df_ex2['Cognoms'] #creem una nova columna que agafa el nom i els cognoms dels usuaris i els concatena en una sola columna.

df_ex2['Es Espanyol'] = df_ex2["País d'origen"] == 'Espanya' #creem una nova columna que ens digui si el usuari ha nascut o no a Espanya. 


df_ex2 = df_ex2.set_index('DNI') #ara substituim el índex numèric pel DNI.  

df_ex2 = df_ex2.rename(columns = { #fem un rename() dels noms de les columnes.
    'Dia de Naixement': 'Dia',
    'Mes de Naixement': 'Mes',
    'Any de Naixement': 'Any'})

df_ex2['Gènere'] = df_ex2['Gènere'].replace({ #fem un replace dels valors possibles de la columna Gènere.
    'H': 'Home',
    'D': 'Dona',
    'A': 'Altres',
    'NC': 'np.nan'
})

df_ex2 #tots els canvis estan desats en la mateixa taula.


# Exercici 3: 
Junta les columnes Fills i No Fills en una sola columna, utilitzant el métode .apply() i definint una funció que resolgui el problema. La columna nova ha de dir-se 'Fills' i prendre els valors 'Sí' o 'No'.

In [ ]:
#Exercici 3

import pandas as pd

df_ex3 = df.copy() #creem una còpia del df original pero no modificar-lo directament. 

#1 Definim la funció
def check_fills(fila): 

    if fila['Fills'] == 1.0:
        return 'Sí'
    elif fila['No Fills'] == 1.0:
        return 'No'
    else: 
        return 'Naan'
    
#2 L'apliquem per crear la nova columna i eliminem les antigues
df_ex3['Fills_Check'] = df_ex3.apply(check_fills, axis=1)
df_ex3 = df_ex3.drop(columns =['Fills', 'No Fills'])

#3 Renombrem la nova columna a 'Fills'
df_ex3 = df_ex3.rename(columns = {'Fills_Check': 'Fills'})

df_ex3



# Exercici 4: 
* Crea una taula resum que permeti veure el sou mig, medià, mínim i màxim per Gènere. 
* Ordena la taula en funció del sou mig.

In [ ]:
#Exercici 4

import pandas as pd

df_ex4 = df.copy()

#0 Netegem la columna: treiem '$', punts de milers i canviem coma decimal per punt
df_ex4['Salari mensual'] = df_ex4['Salari mensual'].astype(str).str.replace('€', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.strip()

#1 Convertim el salari a numèric (treient caràcters si cal, ex: '$')
df_ex4['Salari mensual'] = pd.to_numeric(df_ex4['Salari mensual'], errors='coerce')

#2 Creem la taula resum 
resum_salaris = df_ex4.groupby('Gènere')['Salari mensual'].agg(['mean', 'median', 'min', 'max'])

#3 Ordenem pel sou mig (mean) de forma descendent
resum_salaris = resum_salaris.sort_values(by='mean', ascending=False)

print(resum_salaris.round(2))

# Exercici 5: 
* Crea una taula resum amb el salari mig per gènere (files) i país d'origen (columnes).
* Afegeix-hi les mitjanes als marges de la taula.
* Aplica format condicional a la taula per veure en un color més intens els valors més elevats.

In [ ]:
#Exercici 5

import pandas as pd

df_ex5 = df.copy()

#0 Netegem la columna: treiem '$', punts de milers i canviem coma decimal per punt
df_ex5['Salari mensual'] = df_ex5['Salari mensual'].astype(str).str.replace('€', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).str.strip()

#1 Convertim el salari a numèric (treient caràcters si cal, ex: '$')
df_ex5['Salari mensual'] = pd.to_numeric(df_ex5['Salari mensual'], errors='coerce')

#2 Creem la taula resum 
resum_salaris_gxp = df_ex5.groupby(['Gènere', 'País d\'origen'])['Salari mensual'].mean().unstack()

#3 Li introdu[i]m les mitjanes totals als marges
resum_salaris_marges = df_ex5.pivot_table(
    index='Gènere', 
    columns="País d'origen", 
    values='Salari mensual', 
    aggfunc='mean',
    margins=True,      # Afegeix els totals als marges
    margins_name='Mitjana Total' # Nom per a la fila/columna de totals
).round(2)

resum_salaris_marges

#4 Instalem jinja2 pel degradat visual. 
# Apliquem un degradat de colors (background_gradient)
# El paràmetre 'cmap' defineix la paleta (per exemple: 'YlGnBu' és de groc a blau)
#pip install jinja2

resum_salaris_estil = resum_salaris_marges.style.background_gradient(cmap='coolwarm')

resum_salaris_estil

# Exercici 6:
* Crea una columna nova que sigui la data de naixement en format Datetime a partir de les columnes dia, mes i any. 
* Utilitzant aquesta columna crea una funció que donada una data, et calculi l'edat actual a dia d'avuí. 
* Utilitza la funció que acabes de crear per generar una columna nova al DF amb l'edat actual.

In [ ]:
# Exercici 6
import pandas as pd

df_ex6 = df.copy()

#1 Creem la nova columna 'Data Naixement' - Pandas necessita que els noms de les columnes estiguin en anglès.
# així que per fer la conversió automàtica primer li pasem el diccionari.

df['Data Naixement'] = pd.to_datetime(pd.DataFrame({
    'year': df['Any de Naixement'], 
    'month' : df['Mes de Naixement'], 
    'day' : df['Dia de Naixement']
}))

#2 Comprovem
#print(df[['Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'Data Naixement']].head())

#3 Ara crearem una funció que, mitjansant la columna 'Data Naixement' calculi la edat de cada usuari.
from datetime import date

def calcular_edat(naixement):

    if pd.isnull(naixement):
        return None
    avui = date.today() #Agafa la data d'avuí

    edat = avui.year - naixement.year #Calculem l'edat bàsica en years

    ha_fet_aniversari = (avui.month, avui.day) >= (naixement.month, naixement.day) #Comprovem si ha complert anys aquest any o encara no; comparem si el mes / dia de naixement és superior o inferior al mes d'avui.

    if not ha_fet_aniversari:
        edat -=1
    return edat

#4 Apliquem la funció a la columna de dates que hem creat abans

df['Edat'] = df['Data Naixement'].apply(calcular_edat)

#5 Comprovem 
#print(df[['Data Naixement', 'Edat']].head())

#6 Introdu[i]m la edat al DF 
#6.1 Apliquem la funció a la columna 'Data Naixement'
# que hem creat i guardem el resultat directament en una nova columna 'Edat'.

df_ex6['Edat'] = df_ex6['Data Naixement'].apply(calcular_edat)

#6.2 Visualitzem les dades per confirmar que tot és correcte.
df_ex6

,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Nom complet,Data Naixement,Edat
0,Inês,Ferreira Silva,16928694K,Portugal,Lisboa,25,2,1953,D,1.144 €,NaN,1.0,Grup B,Inês Ferreira Silva,1953-02-25,73
1,Clara,Sánchez Martínez,27724652S,Espanya,Barcelona,18,3,1996,D,1.253 €,1.0,NaN,Grup A,Clara Sánchez Martínez,1996-03-18,30
2,Fatima,Fassi,38141675A,Marroc,Rabat,6,11,2005,A,1.441 €,1.0,NaN,Grup A,Fatima Fassi,2005-11-06,20
3,Khadija,Bennani Bennani,59157262R,Marroc,Rabat,20,1,1995,D,1.944 €,NaN,1.0,Grup B,Khadija Bennani Bennani,1995-01-20,31
4,Toni,Sánchez García,69630528M,Espanya,Barcelona,9,8,1999,H,1.043 €,NaN,1.0,Grup A,Toni Sánchez García,1999-08-09,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Marta,Ferrer Ferrer,25161375F,Espanya,Sevilla,1,6,1951,D,1.216 €,NaN,1.0,Grup B,Marta Ferrer Ferrer,1951-06-01,74
996,Joan,García,52145541P,Espanya,Sevilla,11,4,1959,H,971 €,NaN,1.0,Grup A,Joan García,1959-04-11,67
997,Laia,Ferrer Martínez,69760120X,Espanya,Barcelona,11,11,1980,D,682 €,NaN,1.0,Grup A,Laia Ferrer Martínez,1980-11-11,45
998,Jordi,García,82947791W,Espanya,Barcelona,23,5,1984,H,1.699 €,1.0,NaN,Grup C,Jordi García,1984-05-23,41
